# 🚀 ONN-DETECT: Engine A (SRMForensicCNN) Training on Google Colab (T4 GPU)
### Paired Consistency Training (DCPT) + Native-Resolution PatchCraft 4x4 Mosaics

This notebook trains and evaluates **Engine A (`SRMForensicCNN`)** for the ONN-DETECT deepfake detection engine on a free Google Colab **NVIDIA T4 GPU** instance (16GB VRAM).

#### Key Highlights:
- **DCPT (arXiv:2604.10102)**: Paired consistency loss on clean vs. degraded views of the same image (`QF=30..90`, resize, blur, noise).
- **Native PatchCraft Mosaics (arXiv:2311.12397)**: L1 gradient-energy tile selection preserves native sensor noise fingerprints.
- **T4 GPU Hardware Optimization**: Uses Automatic Mixed Precision (`--amp` via FP16 Tensor Cores) for 2.5x faster throughput.
- **Zero Android Architecture Change**: Emits standard `srm_forensic_cnn_int8.tflite` compatible with on-device inference.

--- 
## Step 1: Verify Hardware & GPU Acceleration
Let's confirm that Google Colab has allocated an NVIDIA T4 GPU and that PyTorch detects CUDA.

In [ ]:
# Check NVIDIA driver and GPU allocation
!nvidia-smi

import torch
print("=" * 60)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active Device   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Allocated  : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print("=" * 60)

--- 
## Step 2: Mount Google Drive (Persistent Storage)
Mounting Google Drive ensures your checkpoints, evaluation JSON summaries, and datasets persist across Colab session disconnects.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/onn_checkpoints'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
print(f"✅ Google Drive mounted. Checkpoints will be backed up to: {DRIVE_BACKUP_DIR}")

--- 
## Step 3: Clone Repository & Pull Latest Code

In [ ]:
%cd /content
!rm -rf Onn-Detect-
!git clone https://github.com/shubhamiitshk/Onn-Detect-.git
%cd /content/Onn-Detect-
!git log -1 --oneline

--- 
## Step 4: Install Dependencies & Run Verification Self-Test
**Important Note**: We do *not* overwrite Colab's pre-installed CUDA PyTorch. We install the repository helper dependencies (`PyYAML`, `scikit-learn`, `scipy`, `xgboost`, `soundfile`, `datasets`) and immediately run the self-test.

In [ ]:
# Install dependencies without overwriting CUDA PyTorch
!pip install -q PyYAML scipy scikit-learn xgboost soundfile Pillow tqdm datasets ai-edge-litert

# Run the built-in self-test (validates PatchCraft frontend, 8-slice degradation matrix, and low-FPR metrics)
!python ml/scripts/train_srm_dcpt_patchcraft.py --self-test

--- 
## Step 5: Prepare Dataset & Dataset Manifest
The ONN-DETECT training contract requires a `dataset_manifest.json` containing 4 splits (`train`, `val`, `calibration`, `test`) with SHA-256 integrity and group leakage checks.

Select one of the 3 setup options below:

In [ ]:
import os, json, hashlib
from pathlib import Path
from PIL import Image
import numpy as np

# Set DATASET_MODE to:
#   'synthetic' -> Generates an instant synthetic dry-run split (tests T4 AMP & training loop in 30 sec)
#   'drive'     -> Unzips your real dataset from Google Drive
#   'cifake'    -> Downloads CIFAKE from Hugging Face & formats it
DATASET_MODE = 'synthetic'  # @param ['synthetic', 'drive', 'cifake']
DATASET_ROOT = Path('/content/datasets/openfake_visual')

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(1024 * 1024):
            h.update(chunk)
    return h.hexdigest()

def build_manifest(root: Path, source_name='custom_dataset', revision='v1'):
    records = []
    for split in ('train', 'val', 'calibration', 'test'):
        for label, cls in [(0, 'real'), (1, 'fake')]:
            cls_dir = root / split / cls
            cls_dir.mkdir(parents=True, exist_ok=True)
            for p in sorted(cls_dir.glob('*.*')):
                if p.suffix.lower() in ('.jpg', '.png', '.jpeg', '.webp'):
                    rel = p.relative_to(root).as_posix()
                    records.append({
                        'path': rel,
                        'split': split,
                        'label': label,
                        'source_id': rel,
                        'group_id': rel,
                        'sha256': sha256_file(p)
                    })
    manifest = {
        'schema_version': 1,
        'source': source_name,
        'revision': revision,
        'label_convention': 'real=0_fake=1',
        'grouping_verified': True,
        'records': records
    }
    (root / 'dataset_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print(f"✅ Manifest created with {len(records)} records across 4 splits.")

if DATASET_MODE == 'synthetic':
    print("⚡ Generating synthetic pilot dataset for instant T4 training verification...")
    DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    counts = {'train': 64, 'val': 16, 'calibration': 16, 'test': 32}
    for split, n in counts.items():
        for label, cls in [(0, 'real'), (1, 'fake')]:
            d = DATASET_ROOT / split / cls
            d.mkdir(parents=True, exist_ok=True)
            for i in range(n):
                arr = np.random.randint(0, 256, (256, 256, 3), dtype=np.uint8)
                # Inject slight high-frequency variance into synthetic class for discriminability
                if label == 1:
                    arr[::4, ::4, :] = (arr[::4, ::4, :] + 40).clip(0, 255)
                Image.fromarray(arr).save(d / f"{split}_{cls}_{i:04d}.png")
    build_manifest(DATASET_ROOT, source_name='synthetic-mock', revision='colab-t4-v1')

elif DATASET_MODE == 'drive':
    ZIP_PATH = '/content/drive/MyDrive/openfake_visual.zip'  # Adjust path if needed
    print(f"📂 Unzipping dataset from {ZIP_PATH}...")
    !unzip -q {ZIP_PATH} -d /content/datasets/
    if not (DATASET_ROOT / 'dataset_manifest.json').exists():
        build_manifest(DATASET_ROOT, source_name='drive-dataset', revision='v1')

print(f"✅ Dataset ready at: {DATASET_ROOT}")

--- 
## Step 6: Train Full DCPT + 4x4 PatchCraft Mosaic Model (Ablation B)

Runs the flagship ONN-DETECT v8 Engine A configuration:
- **Architecture**: `SRMForensicCNN` (SRM + NPR residual filters + RGB spatial branch)
- **Frontend**: Native `4x4` 32x32 PatchCraft Mosaic (`--patchcraft-mode mosaic`)
- **Loss**: Binary Cross-Entropy + DCPT Bernoulli KL Logit Loss (`0.5`) + DCPT Feature Cosine Loss (`0.5`)
- **Hardware**: T4 Tensor Cores via `--amp` (FP16 Automatic Mixed Precision)

In [ ]:
!python ml/scripts/train_srm_dcpt_patchcraft.py \
    --config ml/configs/srm_cnn_openfake.yaml \
    --data-dir /content/datasets/openfake_visual \
    --output-dir ml/checkpoints/ablation_dcpt_patchcraft \
    --model-name srm_dcpt_patchcraft \
    --patchcraft-mode mosaic \
    --dcpt-logit-weight 0.5 \
    --dcpt-feature-weight 0.5 \
    --epochs 12 \
    --batch-size 32 \
    --amp

--- 
## Step 7: Train DCPT-Only Baseline (Ablation A - Bilinear Resize)

Runs the baseline model using standard bilinear resizing (`--patchcraft-mode off`) to measure the performance gain delivered by native PatchCraft mosaics.

In [ ]:
!python ml/scripts/train_srm_dcpt_patchcraft.py \
    --config ml/configs/srm_cnn_openfake.yaml \
    --data-dir /content/datasets/openfake_visual \
    --output-dir ml/checkpoints/ablation_dcpt_only \
    --model-name srm_dcpt_only \
    --patchcraft-mode off \
    --dcpt-logit-weight 0.5 \
    --dcpt-feature-weight 0.5 \
    --epochs 12 \
    --batch-size 32 \
    --amp

--- 
## Step 8: Compare 8-Slice Degradation Matrix Resilience
Let's compare how both models perform across all 8 degradation conditions:
1. `clean`
2. `whatsapp_512_q85` (Real-world messenger re-compression)
3. `jpeg_qf90`
4. `jpeg_qf70`
5. `jpeg_qf50`
6. `jpeg_qf30` (Extreme compression artifact)
7. `downscale_0_5x`
8. `blur_sigma_1_0`

In [ ]:
import json
import pandas as pd

path_mosaic = Path('ml/checkpoints/ablation_dcpt_patchcraft/srm_dcpt_patchcraft_results.json')
path_only = Path('ml/checkpoints/ablation_dcpt_only/srm_dcpt_only_results.json')

def load_results(path, name):
    if not path.exists():
        return {}
    data = json.loads(path.read_text())
    matrix = data.get('degradation_matrix', {})
    rows = {}
    for cond, metrics in matrix.items():
        rows[cond] = {
            f"{name} AUC": round(metrics['auc'], 4),
            f"{name} EER": round(metrics['eer'], 4),
            f"{name} TPR@1%FPR": round(metrics['tpr_at_fpr_1pct'], 4)
        }
    return rows

res_mosaic = load_results(path_mosaic, 'DCPT+PatchCraft (v8)')
res_only = load_results(path_only, 'DCPT Only')

if res_mosaic:
    combined = {}
    for k in res_mosaic:
        combined[k] = {**res_mosaic.get(k, {}), **res_only.get(k, {})}
    df = pd.DataFrame.from_dict(combined, orient='index')
    display(df)
else:
    print("Results files not found yet. Run Step 6 first.")

--- 
## Step 9: Backup Checkpoints & JSON Reports to Google Drive
Saves all `.pth` model weights, logs, and evaluation reports directly into your Google Drive.

In [ ]:
!mkdir -p /content/drive/MyDrive/onn_checkpoints/
!cp -rv ml/checkpoints/* /content/drive/MyDrive/onn_checkpoints/
print("\n🎉 Training Complete! All checkpoints safely backed up to Google Drive:")
!ls -lh /content/drive/MyDrive/onn_checkpoints/